In [2]:
import boto3
import json
import os
import pandas as pd
import numpy as np
import pprint

In [ ]:
os.environ['AWS_BEARER_TOKEN_BEDROCK'] = "="

In [3]:
file = '2018-09-13--06.02.24.events.csv'
sim_data = pd.read_csv(file, header=4)

sim_data["Info"] = sim_data.get("Info", pd.Series()).astype(str)
sim_data = sim_data.drop(index=[31330, 4816], errors='ignore')

drop_events = ['vaccinated', 'exposed']
clean_sim_data = sim_data[~sim_data['HCV'].isin(drop_events)].copy()

print("HCV Counts (Cleaned)")
print(clean_sim_data['HCV'].value_counts())

HCV Counts (Cleaned)
HCV
susceptible        29700
chronic            14627
recovered          12032
infectiousacute     6670
Name: count, dtype: int64


C:\Users\yysma\AppData\Local\Temp\ipykernel_25524\4088997972.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  sim_data = pd.read_csv(file, header=4)


In [4]:


agent_histories = {
    agent_id: group.sort_values('Time')
    for agent_id, group in clean_sim_data.groupby('Agent')
}

In [9]:
baseline = (
    clean_sim_data
    .sort_values(['Time']) 
    .drop_duplicates(subset='Agent', keep='first')
)
baseline = baseline[baseline['HCV'] == 'susceptible'].copy()
baseline['Injection_Experience'] = baseline['Age'] - baseline['Age_Started']

event_counts = clean_sim_data.groupby('Agent').size()
valid_agents = event_counts[event_counts >= 2].index

baseline = baseline[
    (baseline['Age'] >= 18) &
    (baseline['Daily_injection_intensity'] > 0) &
    (baseline['Agent'].isin(valid_agents))
].copy()

# split by experience
young_eligible = baseline[
    baseline['Injection_Experience'] < 3
].copy()
young_eligible['Age_Group'] = 'Young'

old_eligible = baseline[
    baseline['Injection_Experience'] >= 3
].copy()
old_eligible['Age_Group'] = 'Old'

all_eligible = pd.concat([young_eligible, old_eligible])

N_PER_BUCKET = 7
STRATA_COLS = ['Gender', 'Race', 'Age_Group']

# first stratified sample
stratified_agents = (
    all_eligible
    .groupby(['Gender', 'Race', 'Age_Group'], group_keys=False)
    .apply(lambda x: x.sample(7, random_state=42))
)

used_agents = set(stratified_agents['Agent'])
remaining_pool = all_eligible[~all_eligible['Agent'].isin(used_agents)].copy()

# second stratified sample
extra_agents = (
    remaining_pool
    .groupby(['Gender', 'Race', 'Age_Group'], group_keys=False)
    .apply(lambda x: x.sample(
        n=min(7, len(x)),
        random_state=99
    )))
final_sample = pd.concat([stratified_agents, extra_agents])

C:\Users\yysma\AppData\Local\Temp\ipykernel_25524\2356841148.py:38: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(7, random_state=42))
C:\Users\yysma\AppData\Local\Temp\ipykernel_25524\2356841148.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


In [ ]:
all_eligible.groupby(STRATA_COLS).size()
print(extra_agents['HCV'].value_counts())
print(extra_agents['Agent'].duplicated().sum())
final_sample.groupby(['Gender', 'Race', 'Age_Group']).size().unstack()

HCV
susceptible    224
Name: count, dtype: int64
0


Age_Group        Old  Young
Gender Race                
Female Hispanic   14     14
       NHBlack    14     14
       NHWhite    14     14
       Other      14     14
Male   Hispanic   14     14
       NHBlack    14     14
       NHWhite    14     14
       Other      14     14

In [ ]:
data = final_sample.copy()
rows = []

for v in ["Age", "Age_Started"]:
    rows.append({
        "Variable": v,
        "Mean (SD)": f"{data[v].mean():.2f} ({data[v].std():.2f})",
        "Percent": ""
    })

for c in ["Race", "Gender"]:
    tmp = data[c].value_counts(normalize=True).sort_index()

    for k, v in tmp.items():
        rows.append({
            "Variable": f"{c}: {k}",
            "Mean (SD)": "",
            "Percent": f"{v*100:.1f}%"
        })

behav_map = {
    "Daily_injection_intensity": "Weekly injections",
    "Fraction_recept_sharing": "Receptive sharing",
    "Drug_in_degree": "Syringes received",
    "Drug_out_degree": "Syringes given",
    "HCV_friend_preval": "HCV in network"
}

for col, name in behav_map.items():
    value = data[col]
    if col == "Daily_injection_intensity":
        value = value * 7
    rows.append({
        "Variable": name,
        "Mean (SD)": f"{value.mean():.2f} ({value.std():.2f})",
        "Percent": ""
    })
table1 = pd.DataFrame(rows)

age_rows = table1[table1["Variable"].isin(["Age", "Age_Started"])]
other_rows = table1[~table1["Variable"].isin(["Age", "Age_Started"])]

table1 = pd.concat([other_rows, age_rows], ignore_index=True)

print(table1)

             Variable      Mean (SD) Percent
0      Race: Hispanic                  25.0%
1       Race: NHBlack                  25.0%
2       Race: NHWhite                  25.0%
3         Race: Other                  25.0%
4      Gender: Female                  50.0%
5        Gender: Male                  50.0%
6   Weekly injections  21.89 (19.07)        
7   Receptive sharing    0.26 (0.27)        
8   Syringes received    0.62 (0.99)        
9      Syringes given    0.70 (2.24)        
10     HCV in network    0.10 (0.26)        
11                Age   31.60 (8.19)        
12        Age_Started   26.43 (7.91)        


In [21]:
agent_ids = extra_agents['Agent'].unique()

agent_ids_list = extra_agents['Agent'].tolist()

print(f"Number of unique IDs: {len(agent_ids)}")
print(agent_ids[:10])

Number of unique IDs: 112
[1363458668  618736064 2045502604 1702173449 1447776376 1744176741
 1220931314 1631093824 1206809341 1542231393]


In [28]:
prompt_list = []
for agent_id in agent_ids[:]:
    agent_df = agent_histories[agent_id]
    row = agent_df[agent_df['Event'] == 'activated'].iloc[0].copy()

    race_map = {
    "NHBlack": "Black",
    "NHWhite": "White",
    "Hispanic": "Hispanic",
    "Other": "a race other than Black, White, or Hispanic"
}
   
    cleaned_race = race_map.get(row['Race'])

    prompt = (
        "Chicago drug environment: "
        "Overdose events vary across neighborhoods. "
        "Homelessness is defined as lacking a stable nighttime residence, including unstable or non-permanent living situations.\n\n"

        "Individual Profile (during 2025):\n"
        f"- Age: {round(row['Age'])}\n"
        f"- Gender: {row['Gender']}\n"
        f"- Race: {cleaned_race}\n"
        f"- Housing status: Homeless (≥12 months)\n" #Homeless vs. Stable
        f"- Injection history: Started at age {round(row['Age_Started'])}; no prior smoking or snorting experience; uses opioids without fentanyl\n"
        f"- Injection frequency: ~{round(row['Daily_injection_intensity']*7)} injections per week with mild withdrawal symptoms; ~{int(row['Fraction_recept_sharing']*100)}% involved reusing someone else's syringe\n"
        f"- Network: Receives used syringes from {round(row['Drug_in_degree'])} persons and gives used syringes to {round(row['Drug_out_degree'])} persons; {int(row['HCV_friend_preval']*100)}% HCV prevalence among current injection partners\n\n"
        
        "Scenario: "
        "Over a 6-month period in 2026, this individual is highly confident that illicitly manufactured fentanyl is present in their opioid supply. "
        "A nearby harm reduction service provides clean syringes, naloxone, drug checking, and opioid use disorder treatment. Their engagement with these services may vary. \n\n"        
        "Instructions: "
        "Predict the individual's typical weekly behavior over the past 4 weeks. Report measures 1–6 as a weekly average, measures 7-8 as a proportion (between 0 and 1 inclusive, rounded to 2 decimal places), and measures 9-11 as categorical values:\n\n"
        
        "1. Number of injections\n"
        "2. Number of times they reused a syringe that someone else already used\n" 
        "3. Number of times they gave a used syringe to someone else\n"
        "4. Number of unique injection partners (someone they have injected with at least once in the past 4 weeks)\n"
        "5. Number of times they chose to smoke or snort drugs instead of injecting\n" 
        "6. Number of times they experienced moderate to severe opioid withdrawal\n" 
        "7. Proportion of injection partners currently infected with HCV\n"
        "8. Proportion of injections done in a public setting or shelter\n"
        "9. Predominant neighborhood of drug-use activity (past 4 weeks)\n"
        "10. Current engagement in an opioid use disorder treatment program\n"
        "11. Current HCV infection state\n\n"

        "Output Format: "
        "Return your answer as a single-line JSON object with the following keys. Do not include line breaks or outside text.\n\n"
        "Injections (max 200)\n"
        "Recept_sharing (≤ Injections)\n"
        "Drug_out (≤ Injections)\n"
        "Partners\n"
        "Alt_route\n"
        "Withdrawal (≤ Injections + Alt_route)\n"
        "Partner_HCV_prop\n"
        "Public_prop\n"
        "Neighborhood (north side, central, west side, or south side)\n"
        "Treatment (0=no, 1=yes)\n"
        "HCV (susceptible, acute, chronic, or recovered)"
        )

    prompt_list.append(prompt)  

print(prompt_list[0])

Chicago drug environment: Overdose events vary across neighborhoods. Homelessness is defined as lacking a stable nighttime residence, including unstable or non-permanent living situations.

Individual Profile (during 2025):
- Age: 36
- Gender: Female
- Race: Hispanic
- Housing status: Homeless (≥12 months)
- Injection history: Started at age 26; no prior smoking or snorting experience; uses opioids without fentanyl
- Injection frequency: ~12 injections per week with mild withdrawal symptoms; ~0% involved reusing someone else's syringe
- Network: Receives used syringes from 0 persons and gives used syringes to 1 persons; 0% HCV prevalence among current injection partners

Scenario: Over a 6-month period in 2026, this individual is highly confident that illicitly manufactured fentanyl is present in their opioid supply. A nearby harm reduction service provides clean syringes, naloxone, drug checking, and opioid use disorder treatment. Their engagement with these services may vary. 

Instr

In [29]:
client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

model_id = "openai.gpt-oss-120b-1:0"

responses_list = []

for index, api_text in enumerate(prompt_list):
    current_agent_id = agent_ids[index]
    agent_rows = sim_data[sim_data['Agent'] == current_agent_id]
    if agent_rows.empty:
        continue

    agent_row = agent_rows.iloc[0]
    agent_last_row = agent_rows.iloc[-1]

    valid_hcv_states = ['susceptible', 'chronic', 'recovered']
    prev_hcv = 'susceptible'

    if 'HCV' in agent_rows.columns:
        for hcv_state in reversed(agent_rows['HCV'].tolist()):
            if hcv_state in valid_hcv_states:
                prev_hcv = hcv_state
                break
    messages = [{
        "role": "user",
        "content": [{"text": api_text}]
    }]

    response = client.converse(
        modelId=model_id,
        messages=messages,
        inferenceConfig={
            "temperature": 0,
        }
    )
    raw_message_text = ""
    try:
        content_blocks = response.get('output', {}).get('message', {}).get('content', [])

        texts = []
        for block in content_blocks:
            if isinstance(block, dict):
                if 'text' in block:
                    texts.append(block['text'])
                elif block.get('type') in ['text', 'output_text'] and 'text' in block:
                    texts.append(block['text'])

        raw_message_text = " ".join(texts).strip()
    except Exception as e:
        print("Parsing error:", e)
        print("Full response:", json.dumps(response, indent=2))
        raw_message_text = ""

    try:
        structured_response = json.loads(raw_message_text)
    except json.JSONDecodeError:
        structured_response = raw_message_text

    entry = {
        "Agent": current_agent_id.item() if hasattr(current_agent_id, "item") else current_agent_id,
        "Model": "gpt-oss-120b",
        "Homeless": 1,
        "Age": int(round(agent_row['Age'])),
        "Age_Started": int(round(agent_row['Age_Started'])),
        "Race": agent_row['Race'],
        "Gender": agent_row['Gender'],
        "Zip": str(agent_row['Zip']),
        "Prev_HCV_entry": str(agent_row['HCV']),
        "Prev_HCV_exit": str(prev_hcv),
        "Prev_Injection": int(round(agent_row['Daily_injection_intensity']) * 7),
        "Prev_recept_sharing": round(float(agent_row['Fraction_recept_sharing']), 2),
        "Prev_Drug_in": int(agent_row['Drug_in_degree']),
        "Prev_Drug_out": int(agent_row['Drug_out_degree']),
        "Prev_friend_entry": round(float(agent_row['HCV_friend_preval']), 2),
        "Prev_friend_exit": round(float(agent_last_row['HCV_friend_preval']), 2),
        "response": structured_response
    }

    with open("gpt.jsonl", "a") as f:
        f.write(json.dumps(entry) + "\n")

    responses_list.append(entry)

for index, api_text in enumerate(prompt_list): 
    current_agent_id = id_batches[0][index]
    print(current_agent_id)
    print(type(current_agent_id.item()))

In [ ]:
input_file = "nova.jsonl"
output_file = "novapro_cleaned.jsonl"

with open(input_file, "r") as f_in, open(output_file, "w") as f_out:
    for line in f_in:
        entry = json.loads(line)
        
        response_raw = entry["response"]
        if isinstance(response_raw, str):
            cleaned = response_raw.replace("```json", "").replace("```", "").strip()
            
            try:
                structured_response = json.loads(cleaned)
            except json.JSONDecodeError:
                structured_response = response_raw
            
            entry["response"] = structured_response
        
        f_out.write(json.dumps(entry) + "\n")

In [30]:
flat_data = []
with open("gpt.jsonl", "r") as f:
    for line in f:
        row = json.loads(line)  
        response = row.pop("response", "{}")  
        if isinstance(response, str):
            try:
                response = json.loads(response)  
            except json.JSONDecodeEarror:
                response = {}  
        flat_data.append({**row, **response})  # merge

pd.DataFrame(flat_data).to_csv("gpt_output.csv", index=False)

In [ ]:
#print(stratified_agents[["Agent", "Age", "Age_Started", "Daily_injection_intensity"]].to_string(index=False))